In [6]:
# Imports
import os
import sqlite3
from datetime import datetime, timedelta

In [7]:
DB_FILE = "crm_database.db"

# Declaring the function responsible for creating and connecting to the SQLite database.
def create_database():
    conn = None
    try:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()
        print(f"Connected to the database: {DB_FILE}")

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS tb_clients (
            customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            email TEXT UNIQUE,
            phone TEXT,
            company TEXT,
            status TEXT CHECK(status IN ('Lead', 'Active', 'Inactive', 'Prospect')) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """)

        print("Tabela 'tb_clients' verificada/criada.")

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS tb_interactions (
            interaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_id INTEGER NOT NULL,
            interaction_date TIMESTAMP NOT NULL,
            type TEXT CHECK(type IN ('Email', 'Call', 'Meeting', 'Note')) NOT NULL,
            notes TEXT,
            FOREIGN KEY (customer_id) REFERENCES tb_clients (customer_id)
        )
        """)

        print("Tabela 'tb_interactions' verificada/criada.")

        conn.commit()
        print("Database structure ready.")
        return conn, cursor

    except sqlite3.Error as e:
        print(f"Error creating/connecting to database: {e}")
        if conn:
            conn.close()
        return None, None

In [8]:
# Declares the function that populates the tables with sample data.
def populate_tables(conn, cursor):
    print("Populating with example data...")

    try:
        tb_clients_data = [
            ('João Smith', 'john.smith@email.com', '11-9999-0001', 'Empresa Alpha', 'Active'),
            ('Mary Henderson', 'mary.o@sample.net', '21-8888-0002', 'Serviços Beta', 'Active'),
            ('Peter Stone', 'peter.stone@example.org', '31-7777-0003', 'Consultoria Gama', 'Lead'),
            ('Anne Rock', 'ana.rock@email.com', '41-6666-0004', 'Empresa Alpha', 'Inactive'),
            ('Carl Brown', 'carl.brown@sample.net', '51-5555-0005', 'Tec Delta', 'Prospect')
        ]
        inserted_tb_clients = 0
        for customer in tb_clients_data:
            try:
                cursor.execute("""
                INSERT INTO tb_clients (name, email, phone, company, status)
                VALUES (?, ?, ?, ?, ?)
                """, customer)
                
                inserted_tb_clients += 1
            except sqlite3.IntegrityError:
                print(f"Cliente com email {customer[1]} já existe. Ignorando.")

        print(f"{inserted_tb_clients} new clients inserted.")
        cursor.execute("SELECT customer_id, email FROM tb_clients")
        customer_map = {email: cid for cid, email in cursor.fetchall()}
        today = datetime.now()

        tb_interactions_data = [
            (customer_map.get('john.smith@email.com'), (today - timedelta(days=10)).strftime('%Y-%m-%d %H:%M:%S'), 'Call', 'On the first call, he showed interest.'),
            (customer_map.get('john.smith@email.com'), (today - timedelta(days=5)).strftime('%Y-%m-%d %H:%M:%S'), 'Email', 'He sent a commercial proposal.'),
            (customer_map.get('mary.o@sample.net'), (today - timedelta(days=20)).strftime('%Y-%m-%d %H:%M:%S'), 'Meeting', 'Initial introductory meeting.'),
            (customer_map.get('mary.o@sample.net'), (today - timedelta(days=2)).strftime('%Y-%m-%d %H:%M:%S'), 'Email', 'Follow-up after the meeting.'),
            (customer_map.get('peter.stone@example.org'), (today - timedelta(days=1)).strftime('%Y-%m-%d %H:%M:%S'), 'Note', 'Lead captured via website form.'),
            (customer_map.get('carl.brown@sample.net'), (today - timedelta(days=3)).strftime('%Y-%m-%d %H:%M:%S'), 'Email', 'Initial contact sent.')
        ]

        valid_tb_interactions = [inter for inter in tb_interactions_data if inter[0] is not None]
        if valid_tb_interactions:

            cursor.executemany("""
            INSERT INTO tb_interactions (customer_id, interaction_date, type, notes)
            VALUES (?, ?, ?, ?)
            """, valid_tb_interactions)

            print(f"{len(valid_tb_interactions)} embedded interactions.")
        else:
            print("No valid interaction to insert (check if clients were inserted).")

        conn.commit()
        print("Sample data successfully inserted.")

    except sqlite3.Error as e:
        print(f"Error inserting sample data: {e}")
        conn.rollback()

In [9]:
# The script's main function is to create and populate the database.
def main():
    if os.path.exists(DB_FILE):
        print(f"Database '{DB_FILE}' already exists.")

    conn, cursor = create_database()
    if conn and cursor:
        populate_tables(conn, cursor)
        conn.close()
        print("Connection to the database closed.")

In [11]:
print("\nStarting the creation of the CRM database.\n")
main()
print("\nDatabase created successfully.\n")


Starting the creation of the CRM database.

Connected to the database: crm_database.db
Tabela 'tb_clients' verificada/criada.
Tabela 'tb_interactions' verificada/criada.
Database structure ready.
Populating with example data...
5 new clients inserted.
6 embedded interactions.
Sample data successfully inserted.
Connection to the database closed.

Database created successfully.

